# Data Generation
Generate synthetic trade flow data for forecasting analysis.

In [2]:
import pandas as pd
import numpy as np
import os

# 1. Establish robust file paths
NOTEBOOK_DIR = os.getcwd()
OUTPUT_DATA_PATH = os.path.join(NOTEBOOK_DIR, '../data/processed/daily_port_cargo_generation.csv')

# 2. Define Daily Chronological Parameters (June 2023 to June 2026)
# Changing freq="D" expands our baseline to a dense daily timeline
date_range = pd.date_range(start="2023-06-01", end="2026-06-01", freq="D")
total_days = len(date_range)

np.random.seed(42) # Keeps data variance reproducible

# Daily base growth trend (scaling down the linear corporate port growth per day)
base_trend = np.linspace(start=10.5, stop=14.8, num=total_days) / 30.0

cargo_records = []

for idx, current_date in enumerate(date_range):
    current_month = current_date.month
    current_year = current_date.year
    
    # Apply West-Coast India Seasonal Multipliers at a daily grain
    if current_month in [6, 7, 8]:    # Monsoon Slowdown (-18%)
        season_multiplier = 0.82
    elif current_month in [10, 11, 12]: # Q4 Pre-Holiday Peak (+28%)
        season_multiplier = 1.28
    else:
        season_multiplier = 1.00       # Standard baseline
        
    # Standard terminal allocations for Hazira/Mundra
    commodities = [
        {'type': 'Container Tonnage', 'share': 0.45, 'noise_level': 0.05},
        {'type': 'Dry Bulk (Coal/Minerals)', 'share': 0.35, 'noise_level': 0.07},
        {'type': 'Liquid Cargo (POL/Crude)', 'share': 0.20, 'noise_level': 0.03}
    ]
    
    for comm in commodities:
        # Core Daily Formula with Gaussian Noise
        calculated_volume = (base_trend[idx] * comm['share']) * season_multiplier
        random_noise = np.random.normal(0, comm['noise_level'])
        
        # Enforce realistic bounds
        final_volume_mmt = round(max(0.02, calculated_volume + random_noise), 3)
        
        # Split into import/export streams
        import_ratio = 0.55 if comm['type'] == 'Container Tonnage' else 0.48
        import_volume = round(final_volume_mmt * import_ratio, 3)
        export_volume = round(final_volume_mmt * (1 - import_ratio), 3)
        
        # Model daily vessel arrivals using a Poisson distribution (ideal for counts)
        vessel_count = int(np.random.poisson(lam=2 if comm['type'] == 'Container Tonnage' else 1))
        avg_berth_days = round(float(np.random.uniform(1.0, 2.5)), 2) if vessel_count > 0 else 0.0
        
        cargo_records.append({
            "Record_Date": current_date.strftime("%Y-%m-%d"),
            "Year": current_year,
            "Month": current_month,
            "Day": current_date.day,
            "Port_Region": "Gujarat Coast (Hazira/Mundra)",
            "Cargo_Type": comm['type'],
            "Import_Volume_MMT": import_volume,
            "Export_Volume_MMT": export_volume,
            "Total_Volume_MMT": final_volume_mmt,
            "Vessel_Arrival_Count": vessel_count,
            "Avg_Berth_Turnaround_Days": avg_berth_days
        })

# 3. Save to Raw folder
generated_df = pd.DataFrame(cargo_records)
os.makedirs(os.path.dirname(OUTPUT_DATA_PATH), exist_ok=True)
generated_df.to_csv(OUTPUT_DATA_PATH, index=False)

print("🚀 Volume Expansion Complete!")
print(f"📐 New Dataset Dimensions: {generated_df.shape[0]} daily records across {generated_df.shape[1]} features.")

🚀 Volume Expansion Complete!
📐 New Dataset Dimensions: 3291 daily records across 11 features.
